# Can-annular combustor: a staged ring with interconnector cross-flow

A whole can-annular combustor, loaded from a saved case and solved as one reacting network.
Compressor-discharge air enters a common casing **plenum** and is distributed to a **ring of eight reacting cans**, each burning **Jet-A(L)**; the flame tubes are cross-linked by **interconnector** tubes, and the products are collected and accelerated through a **choked nozzle-guide-vane (NGV) throat** on the way to the turbine.

Per can the flow reads: the distributor splits air into a dome/swirl stream and an outer-annulus stream; the swirl air passes a dome cooling-film port and the swirler, meets the fuel, and dumps into the flame tube; an `equilibrium_flame` burns the rich primary mixture; then liner-cooling and dilution air (metered through the annulus and the liner holes) mix into the flame-tube gas, which tapers to the can exit.
The air is not split by hand: each share is set by the hole and swirler resistances and recovered from the solve.

The case is **staged**: the eight cans are fuelled unequally (a smooth rich-to-lean pattern around the ring).
That gives the cans different temperatures and pressures, which drives a real **cross-flow through the interconnector ring** from the hot cans to the cold ones, the feature this case showcases.

The interconnection between the cans form a circular arrangement, forming an internal loop within the network.
This is one of the challenging scenarios for the flow network solver to handle, and as long as the circular flow path provides a loss mechanism the problem is well-posed.

The SVG output of the Network topology from Nemo is shown below.
![Network topology](can_annular_combustor_topology.svg)


## Load and solve

The whole combustor is one saved case; `nefes.load_case` reads the topology and every parameter, and `solve()` finds the reacting mean flow.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.chem import equivalence_ratio_mixture
from nefes.plotting import COLORWAY, use_nefes_theme

use_nefes_theme()

net = nefes.load_case("can_annular_combustor.yaml")

In [ ]:
# Display a summary of the Network setup
net

In [ ]:
sol = net.solve(verbose=2)
assert sol.converged

## Where the delivered air goes

The natural output of a secondary-air model is *how the delivery flow splits*, and here that split is solved, not assumed.
Every branch is metered off the one `plenum`, so the whole air budget closes against the single inlet: the turbine-cooling bypass plus the combustor air (dome, liner cooling, dilution) equals the delivered air.

In [ ]:
mdot = sol.field("mdot")

bypass = mdot[net.edge_between("bypass-pipe", "turbine-cooling")]
dome = sum(mdot[net.edge_between(f"dome-pipe-{i}", f"sw-split-{i}")] for i in range(1, 9))
liner = sum(mdot[net.edge_between(f"dome-split-{2 * i}", f"liner-perf-{i}")] for i in range(1, 9))
dilution = sum(mdot[net.edge_between(f"dome-split-{2 * i}", f"outer-duct-{2 * i}")] for i in range(1, 9))
combustor = dome + liner + dilution
delivered = bypass + combustor

print(f"delivered air              {delivered:6.2f} kg/s")
print(f"  turbine-cooling bypass   {bypass:6.2f} kg/s ({100 * bypass / delivered:2.0f}%)")
print(f"  combustor                {combustor:6.2f} kg/s ({100 * combustor / delivered:2.0f}%)")
print(f"    dome / swirl  (8 cans) {dome:6.2f} kg/s ({100 * dome / combustor:2.0f}% of combustor air)")
print(f"    liner cooling (8 cans) {liner:6.2f} kg/s ({100 * liner / combustor:2.0f}%)")
print(f"    dilution      (8 cans) {dilution:6.2f} kg/s ({100 * dilution / combustor:2.0f}%)")

## The staged ring and the interconnector cross-flow

Each can is fuelled to a different rich primary equivalence ratio, so the cans sit at different flame temperatures.
A hotter can runs at a slightly higher primary-zone pressure than a cooler neighbour, and that small pressure difference pushes gas through the interconnector tube that links them.
The table reads each can's staged fuel, its primary equivalence ratio, its exit temperature, and the net cross-flow leaving it through the interconnector: the sign shows flow running from the hot, rich cans toward the cool, lean ones around the ring.

In [ ]:
T = sol.field("T")

# Compute the stoichiometric mixture
air = {"O2": 0.2095, "N2": 0.7808, "Ar": 0.0093, "CO2": 0.0004}
_st = equivalence_ratio_mixture({"Jet-A(L)": 1.0}, air, 1.0, basis="mass", species_set=net.gas.species_set)

# Compute the stoichiometric fuel mixture fraction
f_stoich = _st["Jet-A(L)"] / (1.0 - _st["Jet-A(L)"])

print("can   fuel [g/s]   primary phi   exit T [K]   interconnector [kg/s]")
for i in range(1, 9):
    fuel = net.get(f"fuel-{i}.mdot")
    air_pz = mdot[net.edge_between(f"dome-cooling-{i}", f"duct-1-cc-{2 * i - 1}")]  # air reaching the flame
    phi = (fuel / air_pz) / f_stoich
    t_exit = T[net.edge_between(f"taper-{i}", "ngv-collector")]
    cross = mdot[net.edge_between(f"ic-{i}", f"duct-ic-{i}")]
    print(f"{i:2d}      {fuel * 1e3:6.1f}       {phi:5.2f}        {t_exit:6.0f}         {cross:+.3f}")

## Axial profile through one can

Following the richest can from the dome to the collector: the flame jumps to its rich, oxygen-limited equilibrium (hot, rich in CO and H$_2$); the liner and dilution air then complete the combustion and cool the gas toward the turbine-inlet value.

In [ ]:
stations = [
    ("sw-split-1", "sw-in-1", "dome air"),
    ("fuel-1", "dump-1", "+ fuel"),
    ("flame-1", "pz-cool-1", "flame"),
    ("pz-cool-1", "duct-1-cc-2", "primary"),
    ("dil-1", "taper-1", "+ dilution"),
    ("taper-1", "ngv-collector", "exit"),
]

keys = ["CO", "CO2", "O2", "H2O"]
palette = {"CO": COLORWAY[4], "CO2": COLORWAY[2], "O2": COLORWAY[0], "H2O": COLORWAY[1]}
labels = [label for _, _, label in stations]
x_ax = list(range(len(stations)))
temps = [T[net.edge_between(a, b)] for a, b, _ in stations]
prof = {k: [sol.species(net.edge_between(a, b), "mass").get(k, 0.0) for a, b, _ in stations] for k in keys}

fig = make_subplots(specs=[[{"secondary_y": True}]])
for k in keys:
    fig.add_scatter(x=x_ax, y=prof[k], name=k, mode="lines+markers", line=dict(color=palette[k]))
fig.add_scatter(
    x=x_ax,
    y=temps,
    name="T",
    mode="lines+markers",
    secondary_y=True,
    line=dict(color="black", dash="dash", width=3),
)
fig.update_xaxes(tickmode="array", tickvals=x_ax, ticktext=labels)
fig.update_yaxes(title_text="mass fraction", secondary_y=False)
fig.update_yaxes(title_text="static temperature [K]", secondary_y=True)
fig.update_layout(title="Axial profile through can 1 (richest primary)")
fig.show()

## The choked NGV sets the combustor pressure

The nozzle-guide-vane throat runs **choked**: the flow reaches Mach 1 there, so the throat area fixes the mass flow leaving the combustor.
With the delivered air held fixed, shrinking the throat backs the pressure up through the whole combustor and pulls a larger share of the delivery flow through the cans (less through the bypass); opening it lets the pressure fall.
`nefes.parameter_study` walks the throat area on pristine `with_params` copies, warm-chaining each solve from the last converged state.

In [ ]:
throats = np.linspace(0.0050, 0.0065, 3)

res = nefes.parameter_study(
    net,
    {"ngv.throat_area": throats},
    probe=lambda s: {
        "p_plenum": float(s.field("p")[net.edge_between("delivery-pipe", "plenum")] / 1e5),
        "bypass_share": float(100 * s.field("mdot")[net.edge_between("bypass-pipe", "turbine-cooling")] / 9.0),
    },
    keep_solutions=False,
    progress=True,
)
print(res)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(
    x=throats * 1e4,
    y=res.probes["p_plenum"],
    mode="lines+markers",
    name="plenum pressure",
    line=dict(color=COLORWAY[0]),
)
fig.add_scatter(
    x=throats * 1e4,
    y=res.probes["bypass_share"],
    mode="lines+markers",
    name="bypass share",
    secondary_y=True,
    line=dict(color=COLORWAY[2]),
)
fig.update_xaxes(title_text="NGV throat area [cm^2]")
fig.update_yaxes(title_text="plenum static pressure [bar]", secondary_y=False)
fig.update_yaxes(title_text="bypass share of delivered air [%]", secondary_y=True)
fig.update_layout(title="Choked NGV throat sets combustor pressure and bypass split")
fig.show()